# diagonal-via-strides — ex2: k-th super-diagonal via as_strided + storage_offset

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `diagonal-via-strides`. Running the final beacon cell reports progress against the `Numpy: Diagonal via strides` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Diagonal via strides` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`diagonal-via-strides`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "diagonal-via-strides"
DD_SUBTOPIC = "Numpy: Diagonal via strides"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## k-th off-diagonal via `as_strided` + storage_offset — quick refresher

The MAIN diagonal of a contiguous `(N, N)` tensor uses `as_strided(size=(N,), stride=(N+1,))` with implicit offset 0. The **k-th SUPER-diagonal** (k > 0) reuses the SAME stride `N+1` and adds the third argument `storage_offset=k`:

```
m.as_strided(size=(N - k,), stride=(N + 1,), storage_offset=k)
```

**Why `storage_offset=k`.** The k-th super-diagonal's first element is `m[0, k]`, which lives at linear offset `k` in row-major storage.

**Why length `N - k`.** Each diagonal step advances both row and col by one — the super-diagonal hits the right wall first; only `N - k` steps stay inside the matrix.

**Exemplar.** For a 4×4 matrix and `k=1`, the super-diagonal is `[m[0,1], m[1,2], m[2,3]]` (length 3, offset 1, stride 5).

### Exercise 2 — k-th super-diagonal via as_strided + storage_offset

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `as_strided(size=(N-k,), stride=(N+1,), storage_offset=k)` to extract the k-th super-diagonal of a contiguous `(N, N)` tensor as a no-copy view, and verify against `torch.diagonal(m, offset=k)`.
> Keywords: as_strided, storage_offset, off-diagonal, super-diagonal
> ```

**KCs targeted:** `diagonal-stride-formula`, `as-strided-storage-offset`

Implement `ex2_kth_super_diagonal(m, k)`. Given a 2-D contiguous tensor `m` of shape `(N, N)` and integer `k` with `0 <= k < N`, return a 1-D length-`(N - k)` view that aliases the k-th super-diagonal of `m`: `[m[0, k], m[1, k+1], ..., m[N-1-k, N-1]]`.

**Use `as_strided` with `storage_offset`.** The stride along the diagonal is still `N + 1` (one row down + one col right). What changes for `k > 0` is the START — the first element lives at linear offset `k` in storage:

```
m.as_strided(size=(N - k,), stride=(N + 1,), storage_offset=k)
```

**The view must alias `m`.** Writing through the returned tensor mutates the k-th super-diagonal of `m`. The test verifies with `.data_ptr()` (now offset from `m.data_ptr()` by `k * element_size`) AND with an in-place write check.

**`k = 0`** must reduce to the main-diagonal case (length `N`, offset `0`).

**Boundary.** Assume `m.is_contiguous()`, `m.dim() == 2`, `m.shape[0] == m.shape[1]`, and `0 <= k < N`.

In [ ]:
def ex2_kth_super_diagonal(m: Tensor, k: int) -> Tensor:
    """Return the k-th super-diagonal as a strided no-copy view."""
    raise NotImplementedError()


def _test_ex2():
    # Hand-checkable 4x4 case, k = 1.
    m = t.arange(16.0).reshape(4, 4).contiguous()
    # m[0,1]=1, m[1,2]=6, m[2,3]=11 → length 3.
    d1 = ex2_kth_super_diagonal(m, k=1)
    assert d1.shape == (3,), f'k=1 length must be N-k=3, got {tuple(d1.shape)}'
    assert t.allclose(d1, t.tensor([1.0, 6.0, 11.0])), f'k=1 values wrong: {d1.tolist()}'

    # k=2: length N-k=2, values m[0,2]=2, m[1,3]=7.
    d2 = ex2_kth_super_diagonal(m, k=2)
    assert d2.shape == (2,)
    assert t.allclose(d2, t.tensor([2.0, 7.0])), f'k=2 values wrong: {d2.tolist()}'

    # k=3: length 1, single element m[0, 3] = 3.
    d3 = ex2_kth_super_diagonal(m, k=3)
    assert d3.shape == (1,)
    assert d3.item() == 3.0

    # k=0 must equal the main diagonal of length N.
    d0 = ex2_kth_super_diagonal(m, k=0)
    assert d0.shape == (4,)
    assert t.allclose(d0, t.tensor([0.0, 5.0, 10.0, 15.0])), f'k=0 must be main diag: {d0}'

    # View — must alias m (data_ptr offset = k * element_size).
    elem = m.element_size()
    for k in [0, 1, 2, 3]:
        v = ex2_kth_super_diagonal(m, k=k)
        expected_ptr = m.data_ptr() + k * elem
        assert v.data_ptr() == expected_ptr, (
            f'k={k}: data_ptr must be m.data_ptr() + {k}*element_size; got offset '
            f'{v.data_ptr() - m.data_ptr()} expected {k * elem}'
        )

    # Write-through aliasing: mutate via view, verify m updated.
    v1 = ex2_kth_super_diagonal(m, k=1)
    v1[1] = -77.0     # was m[1, 2] = 6
    assert m[1, 2].item() == -77.0, 'view must alias — write to v1[1] should update m[1, 2]'
    m[1, 2] = 6.0     # restore

    # Cross-check against torch.diagonal(m, offset=k) on multiple sizes.
    rng = t.Generator().manual_seed(0)
    for N in [2, 3, 5, 8, 16]:
        mk = t.randn(N, N, generator=rng).contiguous()
        for k in range(N):
            ours = ex2_kth_super_diagonal(mk, k=k)
            ref  = t.diagonal(mk, offset=k)
            assert ours.shape == (N - k,) == ref.shape, (
                f'N={N},k={k}: ours shape {tuple(ours.shape)} vs ref {tuple(ref.shape)}'
            )
            assert t.allclose(ours, ref, atol=1e-6), (
                f'N={N},k={k}: ours {ours.tolist()} != t.diagonal {ref.tolist()}'
            )

    # Identity matrix: only the main diagonal is all-ones; every k>0 is all zeros.
    I = t.eye(6)
    assert t.allclose(ex2_kth_super_diagonal(I, k=0), t.ones(6))
    for k in [1, 2, 3]:
        v = ex2_kth_super_diagonal(I, k=k)
        assert v.shape == (6 - k,)
        assert t.allclose(v, t.zeros(6 - k)), f'I, k={k}: must be all zero'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_kth_super_diagonal(m: Tensor, k: int) -> Tensor:
    N = m.shape[0]
    return m.as_strided(size=(N - k,), stride=(N + 1,), storage_offset=k)
```

**`storage_offset` is the third positional / kwarg arg of `as_strided`.** It's the byte offset (in elements, not bytes) from the start of the underlying storage to the first element of the view. The default is `m.storage_offset()` (i.e. start wherever `m` itself starts). Bumping it by `k` skips the first `k` row-major elements — exactly the offset of `m[0, k]`.

**Why the stride is still `N + 1`.** A diagonal-step always moves down-and-right by `(1 row, 1 col) = N + 1` elements in row-major storage. The starting point shifts; the inter-element step does not.

**Sub-diagonals (k < 0).** Outside this drill, but for completeness: the k-th sub-diagonal starts at `m[|k|, 0]`, which lives at offset `|k| * N`. Same stride, length `N - |k|`. `m.as_strided(size=(N - abs(k),), stride=(N + 1,), storage_offset=abs(k) * N)` handles `k < 0`.

**Composability.** Now you can build the full `offdiag_view(m, k)` for any signed `k` in 3 lines — and the sum-of-all-diagonals is a `for k in range(-(N-1), N): sum(...)` loop. No copies anywhere.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()